In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [2]:
!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

os.chdir("RecSys_Course_AT_PoliMi")

!pwd

#!python run_compile_all_cython.py

Cloning into 'RecSys_Course_AT_PoliMi'...
remote: Enumerating objects: 1672, done.
remote: Counting objects: 100% (244/244), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 1672 (delta 221), reused 209 (delta 209), pack-reused 1428 (from 3)
Receiving objects: 100% (1672/1672), 53.49 MiB | 27.51 MiB/s, done.
Resolving deltas: 100% (972/972), done.
/kaggle/working/RecSys_Course_AT_PoliMi


In [3]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender

2025-11-21 11:05:04.296466: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763723104.546715      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763723104.628854      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [4]:
df_train = pd.read_csv("/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv")
df_test_user = pd.read_csv("/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv")

In [5]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [6]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))

In [7]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [8]:

import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function_funksvd(optuna_trial):

                          
    start_time = time.time()
    scores = []
    for i in range(5):
        URM_combined = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        
        
        #Cambiare il modello qui sotto, insieme al range e ai parametri 
        recommender_instance = RP3betaRecommender(URM_combined)
        recommender_instance.fit(alpha = optuna_trial.suggest_float("alpha", 1e-2, 1, log=True),
                             beta = optuna_trial.suggest_float("beta", 1e-2, 1, log=True),
                             topK =  optuna_trial.suggest_int("topK", 1, 150),
                             )
        
        evaluator_test = EvaluatorHoldout(URM_parts[i], cutoff_list=[20])
        result, _ = evaluator_test.evaluateRecommender(recommender_instance)
        #print("prova = ", result["MAP"].values[0])
        #print(result)
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)


In [9]:
import optuna
optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_funksvd,
                      callbacks=[save_results],
                      n_trials = 100)

[I 2025-11-21 11:05:31,886] A new study created in memory with name: no-name-d43083df-77c0-4c62-8ffc-add8f9c4abfd


RP3betaRecommender: Similarity column 6969 (100.0%), 1579.58 column/sec. Elapsed time 4.41 sec
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 14.46 sec. Users per second: 1872
RP3betaRecommender: Similarity column 6969 (100.0%), 1590.41 column/sec. Elapsed time 4.38 sec
EvaluatorHoldout: Ignoring 45 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27050 (100.0%) in 13.92 sec. Users per second: 1944
RP3betaRecommender: Similarity column 6969 (100.0%), 1600.50 column/sec. Elapsed time 4.35 sec
EvaluatorHoldout: Ignoring 39 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27056 (100.0%) in 13.90 sec. Users per second: 1946
RP3betaRecommender: Similarity column 6969 (100.0%), 1578.40 column/sec. Elapsed time 4.42 sec
EvaluatorHoldout: Ignoring 45 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27050 (100.0%) i

[I 2025-11-21 11:07:05,896] Trial 0 finished with value: 0.23403255263402217 and parameters: {'alpha': 0.010083663340292568, 'beta': 0.025848049753720702, 'topK': 80}. Best is trial 0 with value: 0.23403255263402217.


[0.23381683832227484, 0.2338429280740043, 0.23285370655986654, 0.23549304469863286, 0.23415624551533226]
RP3betaRecommender: Similarity column 6969 (100.0%), 1523.55 column/sec. Elapsed time 4.57 sec
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 14.36 sec. Users per second: 1884
RP3betaRecommender: Similarity column 6969 (100.0%), 1530.65 column/sec. Elapsed time 4.55 sec
EvaluatorHoldout: Ignoring 45 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27050 (100.0%) in 14.49 sec. Users per second: 1867
RP3betaRecommender: Similarity column 6969 (100.0%), 1529.93 column/sec. Elapsed time 4.56 sec
EvaluatorHoldout: Ignoring 39 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27056 (100.0%) in 13.93 sec. Users per second: 1943
RP3betaRecommender: Similarity column 6969 (100.0%), 1527.09 column/sec. Elapsed time 4.56 sec
EvaluatorHoldout: Ignor

[I 2025-11-21 11:08:43,841] Trial 1 finished with value: 0.23032437495810215 and parameters: {'alpha': 0.1830346272626715, 'beta': 0.18952663651335397, 'topK': 109}. Best is trial 0 with value: 0.23403255263402217.


...

[0.24725867153988315, 0.24600336368068568, 0.24588337569877702, 0.24802298098373343, 0.24802411869900956]
RP3betaRecommender: Similarity column 6969 (100.0%), 1737.99 column/sec. Elapsed time 4.01 sec
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 13.88 sec. Users per second: 1950
RP3betaRecommender: Similarity column 6969 (100.0%), 1778.72 column/sec. Elapsed time 3.92 sec
EvaluatorHoldout: Ignoring 45 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27050 (100.0%) in 14.10 sec. Users per second: 1919
RP3betaRecommender: Similarity column 6969 (100.0%), 1758.38 column/sec. Elapsed time 3.96 sec
EvaluatorHoldout: Ignoring 39 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27056 (100.0%) in 13.97 sec. Users per second: 1936
RP3betaRecommender: Similarity column 6969 (100.0%), 1789.33 column/sec. Elapsed time 3.89 sec
EvaluatorHoldout: Igno

[I 2025-11-21 13:41:10,620] Trial 98 finished with value: 0.23628476591634012 and parameters: {'alpha': 0.4584391324753137, 'beta': 0.23066594522677075, 'topK': 31}. Best is trial 97 with value: 0.24703850212041778.


[0.2367782923895301, 0.23589171176184778, 0.23570552296416197, 0.2376295940351795, 0.23541870843098128]
RP3betaRecommender: Similarity column 6969 (100.0%), 1720.33 column/sec. Elapsed time 4.05 sec
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 14.35 sec. Users per second: 1886
RP3betaRecommender: Similarity column 6969 (100.0%), 1733.16 column/sec. Elapsed time 4.02 sec
EvaluatorHoldout: Ignoring 45 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27050 (100.0%) in 14.49 sec. Users per second: 1867
RP3betaRecommender: Similarity column 6969 (100.0%), 1732.48 column/sec. Elapsed time 4.02 sec
EvaluatorHoldout: Ignoring 39 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27056 (100.0%) in 14.36 sec. Users per second: 1884
RP3betaRecommender: Similarity column 6969 (100.0%), 1768.62 column/sec. Elapsed time 3.94 sec
EvaluatorHoldout: Ignori

[I 2025-11-21 13:42:45,371] Trial 99 finished with value: 0.24574050367104894 and parameters: {'alpha': 0.5878795556895109, 'beta': 0.41188608061921017, 'topK': 36}. Best is trial 97 with value: 0.24703850212041778.


[0.24524975976065091, 0.2450876567269947, 0.2449151377546512, 0.24624716139296174, 0.247202802719986]


In [10]:
optuna_study.best_trial.params

{'alpha': 0.7733352330682174, 'beta': 0.4139018623121251, 'topK': 35}

In [11]:
save_results.results_df

,result,train_time (min)
0,0.234033,1.566704
1,0.230324,1.632279
2,0.232284,1.568317
3,0.230906,1.646913
4,0.237372,1.563540
...,...,...
95,0.243958,1.588908
96,0.234862,1.768482
97,0.247039,1.569236
98,0.236285,1.531262


In [12]:
best_index = save_results.results_df["result"].idxmax()
best_hyperparams = save_results.results_df.loc[best_index].to_dict()

del best_hyperparams["result"]
del best_hyperparams["train_time (min)"]
best_hyperparams

{}